# Laboratorio 6

Nancy Mazariegos

Santiago Pereira

Brandon Reyes

---


In [1]:
import os
import time
import copy
import random
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, f1_score

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATASET_DIR = Path("MangoLeafBD Dataset")
NUM_CLASSES = 8
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 1e-3
PATIENCE = 4

Device: cpu


In [3]:
def get_transforms(image_size):
    train_tfms = transforms.Compose([
        transforms.Resize((image_size + 32, image_size + 32)),
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    eval_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return train_tfms, eval_tfms

In [4]:
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        if self.transform:
            x = self.transform(x)
        return x, y

base_dataset = datasets.ImageFolder(DATASET_DIR)
class_names = base_dataset.classes
print("Clases:", class_names)
print("Total imágenes:", len(base_dataset))

indices = list(range(len(base_dataset)))
random.shuffle(indices)
indices = indices[:10] 

train_end = int(0.70 * len(indices))
val_end = int(0.85 * len(indices))

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

print(f"Train: {len(train_indices)}")
print(f"Val:   {len(val_indices)}")
print(f"Test:  {len(test_indices)}")

Clases: ['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Powdery Mildew', 'Sooty Mould']
Total imágenes: 4000
Train: 7
Val:   1
Test:  2


In [5]:
def get_dataloaders(image_size, batch_size=32):
    train_tfms, eval_tfms = get_transforms(image_size)
    
    train_dataset = TransformedSubset(base_dataset, train_indices, transform=train_tfms)
    val_dataset = TransformedSubset(base_dataset, val_indices, transform=eval_tfms)
    test_dataset = TransformedSubset(base_dataset, test_indices, transform=eval_tfms)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    
    return train_loader, val_loader, test_loader

In [6]:
def get_model(model_name, num_classes=8):
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        
        for param in model.parameters():
            param.requires_grad = False
        
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        image_size = 224

    elif model_name == "inception_v3":
        weights = models.Inception_V3_Weights.DEFAULT
        model = models.inception_v3(weights=weights)
        
        for param in model.parameters():
            param.requires_grad = False
        
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        
        if model.AuxLogits is not None:
            aux_in_features = model.AuxLogits.fc.in_features
            model.AuxLogits.fc = nn.Linear(aux_in_features, num_classes)
        
        image_size = 299

    elif model_name == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.DEFAULT
        model = models.mobilenet_v2(weights=weights)
        
        for param in model.parameters():
            param.requires_grad = False
        
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        image_size = 224

    else:
        raise ValueError("Modelo no soportado")
    
    return model.to(device), image_size

In [7]:
def train_model(model, train_loader, val_loader, model_name, num_epochs=15, lr=1e-3, patience=4):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0
    
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": []
    }
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            if model_name == "inception_v3":
                outputs, aux_outputs = model(inputs)
                loss1 = criterion(outputs, labels)
                loss2 = criterion(aux_outputs, labels)
                loss = loss1 + 0.4 * loss2
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        train_loss = running_loss / len(train_loader.dataset)
        
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                if model_name == "inception_v3" and isinstance(outputs, tuple):
                    outputs = outputs[0]
                
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping activado.")
                break
    
    model.load_state_dict(best_weights)
    return model, history

In [8]:
def evaluate_model(model, test_loader, model_name):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
            if model_name == "inception_v3" and isinstance(outputs, tuple):
                outputs = outputs[0]
            
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    return acc, f1

In [9]:
def get_model_size_mb(model, filename):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    return size_mb

In [10]:
def measure_inference_time(model, dataset, model_name, num_samples=100):
    model.eval()
    times = []
    
    total = min(num_samples, len(dataset))
    
    with torch.no_grad():
        for i in range(total):
            image, _ = dataset[i]
            image = image.unsqueeze(0).to(device)
            
            start = time.perf_counter()
            outputs = model(image)
            if model_name == "inception_v3" and isinstance(outputs, tuple):
                outputs = outputs[0]
            _ = torch.argmax(outputs, dim=1)
            end = time.perf_counter()
            
            times.append((end - start) * 1000)  
    
    return np.mean(times)

In [11]:
results = {}

model_list = ["resnet50", "inception_v3", "mobilenet_v2"]

for model_name in model_list:
    print("\n" + "="*60)
    print(f"Entrenando {model_name}")
    print("="*60)
    
    model, image_size = get_model(model_name, NUM_CLASSES)
    train_loader, val_loader, test_loader = get_dataloaders(image_size, BATCH_SIZE)
    
    model, history = train_model(
        model,
        train_loader,
        val_loader,
        model_name=model_name,
        num_epochs=NUM_EPOCHS,
        lr=LEARNING_RATE,
        patience=PATIENCE
    )
    
    acc, f1 = evaluate_model(model, test_loader, model_name)
    model_file = f"{model_name}_best.pth"
    size_mb = get_model_size_mb(model, model_file)
    
    _, eval_tfms = get_transforms(image_size)
    test_dataset_for_time = TransformedSubset(base_dataset, test_indices, transform=eval_tfms)
    inf_time = measure_inference_time(model, test_dataset_for_time, model_name, num_samples=100)
    
    results[model_name] = {
        "Accuracy": acc,
        "F1_macro": f1,
        "Size_MB": size_mb,
        "Inference_ms": inf_time
    }
    
    print(f"\nResultados {model_name}:")
    print(results[model_name])


Entrenando resnet50
Epoch [1/3] | Train Loss: 2.1465 | Val Loss: 2.1041 | Val Acc: 0.0000
Epoch [2/3] | Train Loss: 1.9192 | Val Loss: 1.9534 | Val Acc: 0.0000
Epoch [3/3] | Train Loss: 1.8415 | Val Loss: 1.8270 | Val Acc: 0.0000

Resultados resnet50:
{'Accuracy': 0.5, 'F1_macro': 0.3333333333333333, 'Size_MB': 90.03940105438232, 'Inference_ms': np.float64(32.6892499999758)}

Entrenando inception_v3
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /Users/nancymazariegos/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:07<00:00, 13.9MB/s] 


Epoch [1/3] | Train Loss: 2.8289 | Val Loss: 1.9448 | Val Acc: 0.0000
Epoch [2/3] | Train Loss: 2.2857 | Val Loss: 1.8716 | Val Acc: 0.0000
Epoch [3/3] | Train Loss: 1.9809 | Val Loss: 1.7785 | Val Acc: 0.0000

Resultados inception_v3:
{'Accuracy': 0.5, 'F1_macro': 0.3333333333333333, 'Size_MB': 93.27602672576904, 'Inference_ms': np.float64(48.90060449997691)}

Entrenando mobilenet_v2
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /Users/nancymazariegos/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 23.2MB/s]


Epoch [1/3] | Train Loss: 2.1689 | Val Loss: 2.0732 | Val Acc: 0.0000
Epoch [2/3] | Train Loss: 1.9011 | Val Loss: 1.9641 | Val Acc: 0.0000
Epoch [3/3] | Train Loss: 1.7597 | Val Loss: 1.8508 | Val Acc: 0.0000

Resultados mobilenet_v2:
{'Accuracy': 0.0, 'F1_macro': 0.0, 'Size_MB': 8.75464916229248, 'Inference_ms': np.float64(28.873958000019684)}


In [12]:
import pandas as pd

df_results = pd.DataFrame(results).T
df_results = df_results.reset_index().rename(columns={"index": "Model"})
df_results

,Model,Accuracy,F1_macro,Size_MB,Inference_ms
0,resnet50,0.5,0.333333,90.039401,32.689250
1,inception_v3,0.5,0.333333,93.276027,48.900604
2,mobilenet_v2,0.0,0.000000,8.754649,28.873958
